# Nori-Rel quickstart: worked example with outputs

This is the [quickstart](nori_rel_quickstart.ipynb), run once on the public
[RelBench `rel-f1`](https://relbench.stanford.edu/datasets/rel-f1/) Formula 1 database so you can see what every
step prints. Only the settings cells differ from the quickstart, and the optional agent search is turned on. It
predicts each driver's average finishing position over the 60 days after 1 August 2023; a 1 is a win.

The tables are in a Snowflake database you can't read, so this copy is for reading. To run on your own tables, open
the quickstart: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Synthefy/nori-rel/blob/main/colab/nori_rel_quickstart.ipynb)

## Install

If Colab asks you to restart the session after this cell, restart it and go on from the next cell.

In [ ]:
%pip install -q "nori-rel[snowflake,database,databricks,local,agentic]==0.4.7" \
    "synthefy==7.4.0" "synthefy-nori[interpretability]==0.20.1"

In [2]:
import sys
from importlib.metadata import version

import torch

assert (3, 11) <= sys.version_info[:2] <= (3, 13), "Nori-Rel needs Python 3.11 to 3.13"
print("nori-rel", version("nori-rel"), "| synthefy-nori", version("synthefy-nori"))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

nori-rel 0.4.7 | synthefy-nori 0.20.1
GPU: NVIDIA H100 80GB HBM3


## 1. Connection settings

The examples assume two tables: `users(user_id, created_at)` and `orders(order_id, user_id, event_time, amount)`,
with `orders.user_id` a foreign key to `users`.

In [3]:
# "snowflake", "postgresql" or "databricks".
ENGINE = "snowflake"  # example

# Snowflake only.
SNOWFLAKE_WAREHOUSE = "COMPUTE_WH"  # example
SNOWFLAKE_DATABASE = "NORI_REL_DOCS"
SNOWFLAKE_ROLE = None  # example: a role that can read the tables; None uses yours

# Databricks only.
DATABRICKS_CATALOG = "main"  # example

# The schema that holds your tables, and the tables to read (None reads them all).
SCHEMA = "F1"
TABLES = None

# The time column of each event table. Tables without one, such as users, are
# entities. Not sure? Set it to {}, run step 2, and pick from the column list.
TIME_COLUMNS = {
    "races": "date",
    "results": "date",
    "qualifying": "date",
    "standings": "date",
    "constructor_results": "date",
    "constructor_standings": "date",
}

## 2. Connect and check your tables

Prints each table's primary key, time column, links, and columns with their types. Check that every event table has
its time column and that the links you expect are there. Nori-Rel reads the keys your database declares; Snowflake
and Databricks record keys without enforcing them, and Nori-Rel checks them before use. If your tables declare no
keys, pass `primary_keys=` and `foreign_keys=` to `client.connect` (see the
[warehouse guide](https://github.com/Synthefy/nori-rel/blob/main/docs/warehouse-connectors.md)).

In [4]:
import getpass
import json
import os
import re
import warnings

from nori_rel import (
    DatabricksConnection,
    EnvironmentCredential,
    NoriRelClient,
    SnowflakeConnection,
)

# Woodwork and Featuretools warn about type conversions; the warnings don't
# affect results.
warnings.filterwarnings("ignore", module="woodwork")
warnings.filterwarnings("ignore", category=UserWarning, module="featuretools")


def get_secret(name: str, required: bool = True) -> str | None:
    """Return a secret from Colab Secrets, the environment, or a hidden prompt."""
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:  # not in Colab, no such secret, or notebook access is off
        pass
    if os.environ.get(name):
        return os.environ[name]
    return getpass.getpass(f"{name}: ") if required else None


def get_pem(text: str) -> str:
    """Return a PEM key with its line breaks, however it was pasted."""
    pattern = r"(-----BEGIN [A-Z ]+-----)(.*?)(-----END [A-Z ]+-----)"
    match = re.search(pattern, text.replace("\\n", "\n"), re.S)
    if match is None:
        raise ValueError(
            "SNOWFLAKE_PRIVATE_KEY needs the PEM key with BEGIN and END lines"
        )
    body = "".join(match.group(2).split())
    lines = [body[i : i + 64] for i in range(0, len(body), 64)]
    return "\n".join([match.group(1), *lines, match.group(3)]) + "\n"


# Nori-Rel reads each credential from an environment variable of this process.
connection = None
if ENGINE == "snowflake":
    auth = {
        "user": get_secret("SNOWFLAKE_USER"),
        "authenticator": "SNOWFLAKE_JWT",
        "private_key": get_pem(get_secret("SNOWFLAKE_PRIVATE_KEY")),
    }
    if key_password := get_secret("SNOWFLAKE_PRIVATE_KEY_PASSWORD", required=False):
        auth["private_key_password"] = key_password
    os.environ["SNOWFLAKE_AUTH_JSON"] = json.dumps(auth)
    credential = EnvironmentCredential(variable="SNOWFLAKE_AUTH_JSON")
    connection = SnowflakeConnection(
        account=get_secret("SNOWFLAKE_ACCOUNT"),
        warehouse=SNOWFLAKE_WAREHOUSE,
        database=SNOWFLAKE_DATABASE,
        role=SNOWFLAKE_ROLE,
    )
elif ENGINE == "postgresql":
    os.environ["DATABASE_URL"] = get_secret("DATABASE_URL")
    credential = EnvironmentCredential(variable="DATABASE_URL")
elif ENGINE == "databricks":
    os.environ["DATABRICKS_AUTH_JSON"] = get_secret("DATABRICKS_AUTH_JSON")
    credential = EnvironmentCredential(variable="DATABRICKS_AUTH_JSON")
    connection = DatabricksConnection(
        server_hostname=get_secret("DATABRICKS_SERVER_HOSTNAME"),
        http_path=get_secret("DATABRICKS_HTTP_PATH"),
        catalog=DATABRICKS_CATALOG,
    )
else:
    raise ValueError(
        f"ENGINE must be snowflake, postgresql or databricks, not {ENGINE!r}"
    )

client = NoriRelClient(model="nori-30m")
source = client.connect(
    engine=ENGINE,
    connection=connection,
    credential=credential,
    schema_name=SCHEMA,
    tables=TABLES,
    time_columns=TIME_COLUMNS,
)
print("connection:", client.test_connection(source).status.value, "\n")

for table in client.discover(source).tables:
    print(f"{table.name}: key {table.primary_key}, time column {table.time_column}")
    for key in table.foreign_keys:
        print(f"    link {key.columns} -> {key.referred_table}")
    for column in table.columns:
        print(f"    {column.name}: {column.data_type}")

connection: ready 



circuits: key ['circuitId'], time column None
    circuitId: NUMBER
    circuitRef: TEXT
    name: TEXT
    location: TEXT
    country: TEXT
    lat: FLOAT
    lng: FLOAT
    alt: FLOAT
constructor_results: key ['constructorResultsId'], time column date
    link ['constructorId'] -> constructors
    link ['raceId'] -> races
    constructorResultsId: NUMBER
    raceId: NUMBER
    constructorId: NUMBER
    points: FLOAT
    date: TIMESTAMP_TZ
constructor_standings: key ['constructorStandingsId'], time column date
    link ['constructorId'] -> constructors
    link ['raceId'] -> races
    constructorStandingsId: NUMBER
    raceId: NUMBER
    constructorId: NUMBER
    points: FLOAT
    position: NUMBER
    wins: NUMBER
    date: TIMESTAMP_TZ
constructors: key ['constructorId'], time column None
    constructorId: NUMBER
    constructorRef: TEXT
    name: TEXT
    nationality: TEXT
drivers: key ['driverId'], time column None
    driverId: NUMBER
    driverRef: TEXT
    code: TEXT
    forena

## 3. Describe the prediction

- **`QUERY`**: the target in [PQL](https://github.com/Synthefy/nori-rel/blob/main/docs/pql.md). The example is each
  user's total order amount over the 30 days after a date. A count is `COUNT(orders.*,0,30,days)`; a rate divides
  two aggregates, such as `SUM(orders.amount,0,30,days) / COUNT(orders.*,0,30,days)`.
- **`WINDOW`**: the window in `QUERY`.
- **`ANCHOR_TIME`**: the date you predict from.
- **`N_PAST_DATES`**: how many past dates Nori learns from, one `WINDOW` apart before `ANCHOR_TIME`. Each gives one
  example per entity: its features up to that date, and its label from the window after it.
- **`OBSERVED_THROUGH`**: the date your data is complete up to. Labels whose window ends after it are not used.
- **`ENTITY_CREATED_AT`**: the column of the entity table that says when each entity first existed. If your table
  has none, add one, usually each entity's first event time.

In [5]:
from datetime import UTC, datetime, timedelta

QUERY = "PREDICT AVG(results.positionOrder,0,60,days) FOR EACH drivers.driverId"
WINDOW = timedelta(days=60)
ANCHOR_TIME = datetime(2023, 8, 1, tzinfo=UTC)
OBSERVED_THROUGH = ANCHOR_TIME  # example
N_PAST_DATES = 12  # example
ENTITY_CREATED_AT = "dob"
# Cap on candidate rows (entities x past dates), counted before rows without a
# label are dropped. Raise it if step 4 prints few past rows.
MAX_CONTEXT_ROWS = 12_000

# "noridfs" follows links two hops away, within 48 features; "fastdfs" aggregates
# the tables one link away.
FEATURE_STRATEGY = "noridfs"

## 4. Predict

Prints each stage as it runs. The first run also downloads Nori's weights.

In [6]:
request = dict(
    source=source,
    query=QUERY,
    anchor_times=[ANCHOR_TIME - WINDOW * k for k in range(1, N_PAST_DATES + 1)],
    anchor_time=ANCHOR_TIME,
    observed_through=OBSERVED_THROUGH,
    entity_created_at=ENTITY_CREATED_AT,
    max_context_rows=MAX_CONTEXT_ROWS,
    return_details=True,
)
details = client.predict(
    **request,
    feature_strategy=FEATURE_STRATEGY,
    on_event=lambda event: print(f"{event.type}: {event.message}"),
)
print(len(details.predictions), "predictions from", len(details.context), "past rows")
details.predictions.head()

status: Run started: noridfs search.
progress: Started reading the database snapshot.


progress: Finished reading the database snapshot in 28.8 s.
progress: Started planning features.


progress: Finished planning features in 1.2 s.
progress: Started predicting.


progress: Finished predicting in 11.5 s.
status: Run completed.
857 predictions from 223 past rows


,entity_id,anchor_time,prediction
0,0,2023-08-01 00:00:00+00:00,4.915240
1,1,2023-08-01 00:00:00+00:00,13.179745
2,2,2023-08-01 00:00:00+00:00,10.242529
3,3,2023-08-01 00:00:00+00:00,5.776172
4,4,2023-08-01 00:00:00+00:00,13.915490


What went in: one row per entity that gave at least one example, with the labels Nori learned from (oldest past
date first) and its prediction. A blank is a past window that gave no example. Below it, the features, and how
each one was computed.

In [7]:
import pandas as pd

labels = details.context.pivot(index="entity_id", columns="anchor_time", values="label")
history = labels.join(details.predictions.set_index("entity_id")["prediction"])
history.sort_values("prediction", ascending=False).round(2).head(10)

,2021-08-11 00:00:00+00:00,2021-10-10 00:00:00+00:00,2021-12-09 00:00:00+00:00,2022-02-07 00:00:00+00:00,2022-04-08 00:00:00+00:00,2022-06-07 00:00:00+00:00,2022-08-06 00:00:00+00:00,2022-10-05 00:00:00+00:00,2023-02-02 00:00:00+00:00,2023-04-03 00:00:00+00:00,2023-06-02 00:00:00+00:00,prediction
entity_id,,,,,,,,,,,,
856,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.67,18.00,16.50,16.65
851,17.75,18.00,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.58
854,NaN,NaN,NaN,NaN,NaN,NaN,9.00,NaN,14.33,16.67,16.50,16.00
847,13.75,16.00,16.0,17.0,15.4,16.17,17.50,15.8,NaN,NaN,NaN,15.91
824,NaN,NaN,NaN,7.0,15.2,14.33,14.75,15.2,13.33,14.00,17.33,15.02
807,NaN,NaN,NaN,14.5,NaN,NaN,NaN,NaN,11.33,16.33,15.83,15.00
853,NaN,NaN,NaN,10.5,16.2,14.67,15.00,13.0,12.67,16.00,13.50,14.42
850,17.75,14.17,4.0,13.5,12.2,16.67,15.50,14.2,10.67,12.00,14.33,13.97
840,14.00,11.83,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.75


In [8]:
details.get_feature_table().head(10)

,feature,internal_name,source,sql,dtype,missing_share,mean,min,max
0,anchor.entity_key,nr|anchor|entity_key,noridfs,None,object,0.0,NaN,NaN,NaN
1,anchor.cutoff_epoch_days,nr|anchor|cutoff_epoch_days,noridfs,None,float64,0.0,19570.000000,19570.0,19570.0
2,anchor.cutoff_year,nr|anchor|cutoff_year,noridfs,None,float64,0.0,2023.000000,2023.0,2023.0
3,anchor.cutoff_month,nr|anchor|cutoff_month,noridfs,None,float64,0.0,8.000000,8.0,8.0
4,anchor.cutoff_day,nr|anchor|cutoff_day,noridfs,None,float64,0.0,1.000000,1.0,1.0
5,anchor.cutoff_day_of_week,nr|anchor|cutoff_day_of_week,noridfs,None,float64,0.0,1.000000,1.0,1.0
6,drivers.dob,nr|direct|drivers.dob,noridfs,None,float64,0.0,30000.913652,8152.0,46236.0
7,qualifying.count,nr|path|drivers%3C-qualifying.driverId|rows|co...,noridfs,None,float64,0.0,11.452742,0.0,353.0
8,qualifying.constructors.count,nr|path|drivers%7CD%3Aqualifying.driverId-%3Ed...,noridfs,None,float64,0.0,11.452742,0.0,353.0
9,results.count,nr|path|drivers%3C-results.driverId|rows|count...,noridfs,None,float64,0.0,30.431739,1.0,370.0


## 5. Check on a window you already know

The forecast from `ANCHOR_TIME` can't be checked yet. To see how far to trust it, forecast from one `WINDOW` earlier
the same way, using only data up to that date, and compare with what happened. As a reference, it also scores
repeating each entity's previous window. This is one window: check more dates before you rely on a small difference.

In [9]:
check_time = ANCHOR_TIME - WINDOW
check = client.predict(
    **{
        **request,
        "anchor_times": [check_time - WINDOW * k for k in range(1, N_PAST_DATES + 1)],
        "anchor_time": check_time,
        "observed_through": check_time,
    },
    feature_strategy=FEATURE_STRATEGY,
)
scored = check.predictions.set_index("entity_id")[["prediction"]]
# The main forecast learned from the window after check_time: what happened.
scored["actual"] = labels[pd.Timestamp(check_time)]
scored["previous_window"] = labels[pd.Timestamp(check_time - WINDOW)]
scored = scored.dropna()  # entities with a known outcome and a previous window
nori = (scored["prediction"] - scored["actual"]).abs().mean()
repeat = (scored["previous_window"] - scored["actual"]).abs().mean()
print(f"Window after {check_time:%Y-%m-%d}, {len(scored)} entities")
print(f"  mean absolute error, Nori: {nori:.3f}")
print(f"  mean absolute error, repeat the previous window: {repeat:.3f}")

Window after 2023-06-02, 20 entities
  mean absolute error, Nori: 1.739
  mean absolute error, repeat the previous window: 2.133


## 6. Save and replay

`details.pipeline` records how the prediction was made: the task, the features and the model settings. It holds no
credentials, rows or model weights. Replaying it reads the database again and rebuilds the same features, with no
new feature search. To predict a later date, pass that date as `as_of`.

In [10]:
from nori_rel import NoriRelPipeline

details.pipeline.save("pipeline.json")
replayed = client.predict(
    source=source,
    pipeline=NoriRelPipeline.load("pipeline.json"),
    as_of=ANCHOR_TIME,
    observed_through=OBSERVED_THROUGH,
)
same = replayed["prediction"].equals(details.predictions["prediction"])
print(len(replayed), "predictions, identical to the first run:", same)

857 predictions, identical to the first run: True


## 7. Optional: explain one prediction

SHAP splits one entity's prediction into one push per feature, starting from the average prediction. This fits Nori
again on the rows the prediction learned from, in the same order, and checks it gives the same prediction.

In [11]:
import shapiq
from synthefy_nori import NoriRegressor

ENTITY = details.predictions["entity_id"].iloc[0]  # example: any entity_id

columns = details.context_features.columns
output_type = details.metadata.get("output_type") or "median"
output_type = getattr(output_type, "value", output_type)
model = NoriRegressor(model="nori-30m")
model.fit(details.context_features, details.context["label"])


def predict(rows):
    """Return Nori's predictions for feature rows given as an array."""
    frame = pd.DataFrame(rows, columns=columns)
    frame = frame.astype(details.context_features.dtypes.to_dict())
    return model.predict(frame, output_type=output_type)


row = details.features.loc[[ENTITY], columns].to_numpy()
reported = details.predictions.set_index("entity_id").loc[ENTITY, "prediction"]
print(f"entity {ENTITY}: refitted {predict(row)[0]:.3f}, reported {reported:.3f}")
explainer = shapiq.TabularExplainer(
    model=predict,
    data=details.context_features.to_numpy(),
    imputer="baseline",
    index="SV",
    max_order=1,
    random_state=0,
)
sv = explainer.explain(row, budget=1024)
values = {columns[key[0]]: v for key, v in sv.dict_values.items() if len(key) == 1}
print(f"average prediction {sv.baseline_value:.3f}; the features that moved it most:")
for name, value in sorted(values.items(), key=lambda item: -abs(item[1]))[:8]:
    print(f"  {name:40s} {value:+.3f}")

entity 0: refitted 4.920, reported 4.915


average prediction 11.377; the features that moved it most:
  standings.points.mean                    -2.132
  qualifying.number.mean                   -1.222
  results.grid.mean                        -1.006
  qualifying.position.mean                 -0.934
  anchor.entity_key                        -0.907
  drivers.dob                              +0.635
  results.count                            -0.225
  results.mean_age                         +0.200


## 8. Optional: agent feature search

`AgenticSearch` has Claude write SQL features on top of NoriDFS. It keeps them only if they beat NoriDFS on held-out
past rows, and otherwise falls back to NoriDFS. It needs `ANTHROPIC_API_KEY` and costs up to `max_cost_usd`.

The agent's SQL runs here, but the agent sends your schema, column profiles with some raw values, and samples of its
query results to Anthropic. The [`nori-rel` README](https://github.com/Synthefy/nori-rel#readme) lists exactly what
it sends. Set `RUN_AGENT = True` to run it.

In [12]:
RUN_AGENT = True

if RUN_AGENT:
    from nori_rel import AgenticSearch
    from nori_rel.feature.agentic.usage import TokenPrices

    os.environ["ANTHROPIC_API_KEY"] = get_secret("ANTHROPIC_API_KEY")
    search = AgenticSearch.claude(
        model="claude-sonnet-5-5",
        prices=TokenPrices(
            input=3.0,
            output=15.0,
            cache_read=0.3,
            cache_write_5m=3.75,
            cache_write_1h=6.0,
        ),
        max_cost_usd=3.0,  # stop once the agent's token cost reaches $3
    )
    agent_details = client.predict(
        **request,
        feature_strategy=search,
        on_event=lambda event: print(f"{event.type}: {event.message}"),
    )
    keys = ("agentic_trials", "agentic_cost_usd", "agentic_fallback_reason")
    print({key: agent_details.metadata[key] for key in keys})

status: Run started: agentic search, up to 24 trials.
progress: Started reading the database snapshot.


progress: Finished reading the database snapshot in 19.5 s.
progress: Started planning features.


metric: Baseline: MAE 2.639 (48 features).


metric: Agent spend: $0.01 (1 LLM calls).


metric: Agent spend: $0.06 (2 LLM calls).


...

metric: Trial 12/24: MAE 2.182 (best 2.168, baseline 2.639); not a significant gain.


metric: Agent spend: $1.55 (21 LLM calls).
progress: Started confirming the best pipeline.


progress: Finished confirming the best pipeline in 16.6 s.


progress: Finished planning features in 558.0 s.
progress: Started predicting.


progress: Finished predicting in 3.1 s.
status: Run completed.
{'agentic_trials': 12, 'agentic_cost_usd': 1.5501923999999998, 'agentic_fallback_reason': None}


## Good to know

- Each prediction reads every table as of one moment. Snowflake uses Time Travel and Databricks pins Delta table
  versions, so those tables need history. Views are not supported.
- Running Nori here predicts numeric targets. A yes/no target, such as `COUNT(orders.*,0,30,days)=0`, needs hosted
  mode.
- No dates at all? A static target, such as `PREDICT users.income FOR EACH users.user_id`, fills in a column from the
  rows where it is known; see [static targets](https://github.com/Synthefy/nori-rel/blob/main/docs/pql.md#static-targets).
- Your database bills the reads.
- To share results with us, send the check from step 5 and `pipeline.json`. The pipeline holds no credentials or rows.